# SINDy Portfolio Optimization — v8 (Extended History)

**A SINDy-based portfolio optimization system** that uses sparse identification of
nonlinear dynamics for regime detection and adaptive risk management across multiple
asset universes.

---

### Module Structure

| Module | Contents |
|---|---|
| `config.py` | Universe definitions, backtest parameters, OOS split dates |
| `data.py` | `load_returns` — yfinance data fetcher with winsorisation |
| `models.py` | `RiskFeatureExtractor`, `SINDyRegimeDetector`, `SINDyCovarianceEngine` |
| `forecasters.py` | `SampleMean`, `Momentum`, `Zero`, `SINDyRiskMomentum` forecasters |
| `solvers.py` | `solve_mvo`, `solve_cvar` — portfolio optimisers |
| `backtester.py` | `backtest`, `bootstrap_sharpe_diff`, `run_oos`, reporting helpers |
| `visualization.py` | `plot_results` — cumulative returns and weight charts |

### Key Changes from v7

- History extended to **2005** (was 2010) — adds ~1,250 trading days incl. 2008 crisis
- **Two OOS splits**: 2015 (~2,500 OOS days) and 2019 (~1,800 OOS days)
- Universes adjusted for ETF availability pre-2007
- **Goal**: Resolve the Momentum MVO comparison (p = 0.168 in v7)

## 1 — Setup

Install dependencies (run once if needed).

In [ ]:
# pip install numpy pandas cvxpy pysindy yfinance scikit-learn scipy matplotlib

## 2 — Imports

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import sys, os
import numpy as np

# Ensure the parent directory is on the path
sys.path.insert(0, os.path.abspath('..'))

from SINDyFinance import (
    # Config
    UNIVERSES, COMMON, COMMON_OOS, OOS_SPLITS, AF,
    # Data
    load_returns,
    # Forecasters
    MomentumForecaster, SINDyRiskMomentumForecaster,
    # Backtester
    backtest, bootstrap_sharpe_diff, run_oos,
    print_oos, print_bootstrap, max_drawdown,
    # Visualization
    plot_results,
)

print("Imports OK.")

## 3 — Load Universes

In [ ]:
universe_data = {}
for name, cfg in UNIVERSES.items():
    print(f"\nLoading {name} (from {cfg['start']})...")
    rets_u = load_returns(cfg["tickers"], start=cfg["start"])
    if rets_u is not None and len(rets_u) > 1000:
        asset_classes = [cfg["class_map"].get(t, 'equity') for t in rets_u.columns]
        universe_data[name] = {
            "rets": rets_u,
            "asset_classes": asset_classes,
            "tickers": list(rets_u.columns),
        }
        print(f"  {rets_u.shape[0]} days x {rets_u.shape[1]} assets")
        print(f"  {rets_u.index[0].date()} -> {rets_u.index[-1].date()}")
        for t, ac in zip(rets_u.columns, asset_classes):
            print(f"    {t:>5s} -> {ac}")
    else:
        print(f"  SKIPPED")

print(f"\nLoaded {len(universe_data)} universes.")

## 4 — Run All Universes × Two OOS Splits

This cell runs the full-sample and OOS backtests for every universe, generates
cross-universe summaries, and runs the pooled bootstrap to answer the Momentum question.

In [ ]:
all_results = {}

for uname, udata in universe_data.items():
    rets_df = udata["rets"]
    ac = udata["asset_classes"]
    N = rets_df.shape[1]

    print(f"\n{'='*100}")
    print(f"  UNIVERSE: {uname} ({N} assets, {rets_df.shape[0]} days)")
    print(f"  {rets_df.index[0].date()} -> {rets_df.index[-1].date()}")
    print(f"{'='*100}")

    # Full sample
    print(f"\n  Full sample:")
    full = {}

    full["Equal Weight"] = {}
    X = rets_df.values
    T = X.shape[0]
    tm = COMMON["train_min"]
    pnl = X[tm:] @ (np.ones(N) / N)
    c = np.cumprod(1 + pnl)
    mr, vo = pnl.mean() * AF, pnl.std(ddof=1) * np.sqrt(AF)
    down = pnl[pnl < 0]
    full["Equal Weight"] = {
        "pnl": pnl, "curve": c, "sharpe": mr / (vo + 1e-12),
        "mean_ret": mr, "vol": vo, "max_dd": max_drawdown(c),
    }

    print(f"    {'Momentum MVO':<20s} ... ", end="", flush=True)
    try:
        full["Momentum MVO"] = backtest(rets_df,
            forecaster_cls=MomentumForecaster, forecaster_kwargs={"lookback": 60},
            solver="mvo", **COMMON)
        print(f"Sharpe={full['Momentum MVO']['sharpe']:.3f}")
    except Exception as e:
        print(f"FAILED: {e}")

    print(f"    {'SINDy CVaR':<20s} ... ", end="", flush=True)
    try:
        full["SINDy CVaR"] = backtest(rets_df,
            forecaster_cls=SINDyRiskMomentumForecaster,
            forecaster_kwargs={
                "mom_lookback": 60, "regime_threshold_pctile": 75,
                "n_components": 4, "degree": 2,
                "equity_dampen": 0.3, "defensive_boost": 2.0,
                "asset_classes": ac,
            },
            solver="cvar", adaptive_bounds=True, stress_ub_mult=0.6,
            stress_gamma_mult=2.0, **COMMON)
        print(f"Sharpe={full['SINDy CVaR']['sharpe']:.3f}")
    except Exception as e:
        print(f"FAILED: {e}")

    # OOS for each split
    oos_results = {}
    for split in OOS_SPLITS:
        n_oos = len(rets_df[rets_df.index >= split])
        print(f"\n  OOS split at {split} ({n_oos} days):")
        oos = run_oos(rets_df, ac, split, label=f"{uname} @ {split}")
        print_oos(oos)
        print_bootstrap(oos)
        oos_results[split] = oos

    all_results[uname] = {"full": full, "oos": oos_results}

## 5 — Cross-Universe Summary & Pooled Bootstrap

In [ ]:
# ═══════════════════════════════════════════════
# CROSS-UNIVERSE SUMMARY PER SPLIT
# ═══════════════════════════════════════════════
for split in OOS_SPLITS:
    print(f"\n\n{'='*100}")
    print(f"CROSS-UNIVERSE SUMMARY — OOS split at {split}")
    print(f"{'='*100}")

    print(f"\n  {'Universe':<20s} {'SINDy':>7s} {'EW':>7s} {'RP':>7s} {'Mom':>7s} {'Wins':>6s}")
    print(f"  {'-'*55}")

    for uname, ur in all_results.items():
        oos = ur["oos"].get(split, {})
        sc = oos.get("SINDy CVaR", {}).get("sharpe", float("nan"))
        ew = oos.get("Equal Weight", {}).get("sharpe", float("nan"))
        rp = oos.get("Risk Parity", {}).get("sharpe", float("nan"))
        mom = oos.get("Momentum MVO", {}).get("sharpe", float("nan"))
        bms = [v for v in [ew, rp, mom] if not np.isnan(v)]
        wins = sum(1 for b in bms if sc > b) if not np.isnan(sc) else 0
        print(f"  {uname:<20s} {sc:>7.3f} {ew:>7.3f} {rp:>7.3f} {mom:>7.3f} {wins}/{len(bms)}")


# ═══════════════════════════════════════════════
# POOLED BOOTSTRAP — focus on Momentum comparison
# ═══════════════════════════════════════════════
print(f"\n\n{'='*100}")
print(f"POOLED BOOTSTRAP — THE MOMENTUM QUESTION")
print(f"{'='*100}")

for split in OOS_SPLITS:
    pooled_sindy, pooled_mom, pooled_ew, pooled_rp = [], [], [], []
    total_days = 0

    for uname, ur in all_results.items():
        oos = ur["oos"].get(split, {})
        if "SINDy CVaR" in oos:
            pooled_sindy.append(oos["SINDy CVaR"]["pnl"])
            total_days += len(oos["SINDy CVaR"]["pnl"])
        if "Momentum MVO" in oos:
            pooled_mom.append(oos["Momentum MVO"]["pnl"])
        if "Equal Weight" in oos:
            pooled_ew.append(oos["Equal Weight"]["pnl"])
        if "Risk Parity" in oos:
            pooled_rp.append(oos["Risk Parity"]["pnl"])

    if pooled_sindy:
        sindy_pool = np.concatenate(pooled_sindy)
        print(f"\n  Split at {split} (pooled {total_days} OOS days across {len(pooled_sindy)} universes):")

        for bm_name, bm_pool in [("Momentum MVO", pooled_mom), ("Equal Weight", pooled_ew), ("Risk Parity", pooled_rp)]:
            if bm_pool:
                bm_concat = np.concatenate(bm_pool)
                diff, ci, pval = bootstrap_sharpe_diff(sindy_pool, bm_concat)
                sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."
                marker = "  <<<" if bm_name == "Momentum MVO" else ""
                print(f"    vs {bm_name:<20s}: dSharpe={diff:+.4f}  CI[{ci[0]:+.4f},{ci[1]:+.4f}]  p={pval:.3f} {sig}{marker}")

## 6 — Final Verdict

In [ ]:
print(f"\n\n{'='*100}")
print(f"FINAL VERDICT")
print(f"{'='*100}")

# Check the 2015 split (more data) for the Momentum comparison
split_key = "2015-01-01"
mom_results = []
for uname, ur in all_results.items():
    oos = ur["oos"].get(split_key, {})
    if "SINDy CVaR" in oos and "Momentum MVO" in oos:
        sc = oos["SINDy CVaR"]["sharpe"]
        mom = oos["Momentum MVO"]["sharpe"]
        mom_results.append((uname, sc, mom, sc - mom))

if mom_results:
    print(f"\n  SINDy CVaR vs Momentum MVO (OOS from {split_key}):")
    all_win = True
    for uname, sc, mom, diff in mom_results:
        win = "✓" if diff > 0 else "✗"
        print(f"    {uname:<20s}: SINDy {sc:.3f} vs Mom {mom:.3f} (d={diff:+.3f}) {win}")
        if diff <= 0:
            all_win = False

    # Pooled
    pooled_s = np.concatenate([ur["oos"][split_key]["SINDy CVaR"]["pnl"]
                                for uname, ur in all_results.items()
                                if "SINDy CVaR" in ur["oos"].get(split_key, {})])
    pooled_m = np.concatenate([ur["oos"][split_key]["Momentum MVO"]["pnl"]
                                for uname, ur in all_results.items()
                                if "Momentum MVO" in ur["oos"].get(split_key, {})])
    diff, ci, pval = bootstrap_sharpe_diff(pooled_s, pooled_m)
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else "n.s."

    print(f"\n    POOLED: dSharpe={diff:+.4f}  CI[{ci[0]:+.4f},{ci[1]:+.4f}]  p={pval:.3f} {sig}")

    if pval < 0.05:
        print(f"\n  ✓ RESOLVED: SINDy CVaR significantly beats Momentum MVO (p={pval:.3f}).")
        print(f"    The extended history provides enough statistical power.")
    elif pval < 0.10:
        print(f"\n  ~ CLOSE: SINDy CVaR beats Momentum at p={pval:.3f} (marginal).")
        print(f"    Additional data or universes may push this below 0.05.")
    else:
        print(f"\n  ✗ UNRESOLVED: p={pval:.3f}. Still cannot conclusively beat Momentum MVO.")
        print(f"    The advantage is economically meaningful but statistically elusive.")
        print(f"    This is an honest result — report it as an open question.")

## 7 — Visualization

In [ ]:
# Plot results for the 2015 split (more OOS data)
plot_results(all_results, split="2015-01-01")

In [ ]:
# Plot results for the 2019 split
plot_results(all_results, split="2019-01-01")

## Conclusions — v8

**Extended history (2005–2025)** adds ~1,250 days and captures the 2008 crisis.

**Two OOS splits:**
- 2019 split: same as v7, for direct comparison
- 2015 split: ~2,500 OOS days per universe, more statistical power

**The Momentum question:** With the extended history and pooled cross-universe test,
does SINDy CVaR finally reach p < 0.05 against Momentum MVO?

If yes: the paper can claim significant improvement over all benchmarks.
If no: report it honestly. Beating 60/40 and risk parity is still a strong result.